# 15. Project - Low Memory Alerts as Jira Tickets

**Goal:** connect to a Linux server over SSH, check its free memory, and when it's too low, **open a Highest-priority Jira ticket and assign it** to the on-call engineer.

![Low memory alert flow](images/15-memory-alert-flow.png)

The finished script is in [`projects/memory_alert.py`](projects/memory_alert.py).

> Run this from your own machine or a server that can reach the target over SSH. A hosted notebook such as Colab usually can't reach servers on your private network.

### What You Need

| Setting | Environment variable |
|---|---|
| Jira site, email and API token | `JIRA_SERVER`, `JIRA_EMAIL`, `JIRA_API_TOKEN` |
| Server address and SSH user | `SSH_HOST`, `SSH_USER` |
| SSH password (or better, an SSH key) | `SSH_PASSWORD` |
| Account ID of the engineer to assign | `ONCALL_ACCOUNT_ID` |

The project needs an issue type called **Issue** (topic 07). Change `issuetype` to `Task` or `Bug` if your project doesn't have it.

## Step 1: Read the Memory over SSH

In [1]:
!pip install paramiko

In [2]:
import os
import paramiko

hostname = os.environ["SSH_HOST"]
client = paramiko.SSHClient()
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
client.connect(hostname=hostname, username=os.environ["SSH_USER"], password=os.environ["SSH_PASSWORD"])

stdin, stdout, stderr = client.exec_command("free -g")
cmdout = stdout.readlines()
client.close()

print("".join(cmdout))

`free -g` prints memory in gigabytes. Example output:

```text
               total        used        free      shared  buff/cache   available
Mem:               7           5           0           0           1           1
Swap:              1           0           1
```

`cmdout[1]` is the `Mem:` line. Split it on whitespace and the **7th value** (index 6) is **available** memory:

```python
>>> "Mem:  7  5  0  0  1  1".split()
['Mem:', '7', '5', '0', '0', '1', '1']
>>> "Mem:  7  5  0  0  1  1".split()[6]
'1'
```

> Use **available**, not **free**. Linux uses spare RAM as cache, so *free* is often near zero on a healthy server, while *available* is what new programs can actually get.

In [3]:
available_mem = int(cmdout[1].split()[6])
print("Available memory:", available_mem, "GB")

## Step 2: Open a Ticket When Memory Is Low

In [4]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

In [5]:
import time

if available_mem <= 1:
    print("Memory is not good")
    summary = f"Memory available {available_mem}GB and Server name is {hostname}"
    description = f"Hi Team, we found a low memory issue on this server {hostname}.\n{summary}"

    issue_dict = {
        "project": {"key": "IIP"},
        "issuetype": {"name": "Issue"},
        "summary": summary,
        "description": description,
        "priority": {"name": "Highest"},
    }
    new_issue = jira.create_issue(fields=issue_dict)
    time.sleep(3)
    new_issue.update(assignee={"accountId": os.environ["ONCALL_ACCOUNT_ID"]})
    print("Ticket has been created:", new_issue.key)
else:
    print("Memory is good")

### What the Ticket Looks Like

On the practice site, a server with 1 GB available produced a ticket like this:

| Field | Value |
|---|---|
| Key | `IIP-24` |
| Summary | Memory available 1GB and Server name is 192.168.1.6 |
| Description | Hi Team, we found a low memory issue on this server 192.168.1.6. Memory available 1GB and Server name is 192.168.1.6 |
| Priority | Highest |
| Status | To Do |
| Assignee | The on-call engineer |

## Step 3: Make It Production-Ready

- **Don't open a ticket on every run.** Before creating one, search for an open ticket for the same server:

```python
existing = jira.search_issues(f'project = IIP AND summary ~ "{hostname}" AND resolution = Unresolved')
if existing:
    jira.add_comment(existing[0], f"Still low: {available_mem}GB available")
else:
    ...  # create the ticket
```

- **Check several servers** by looping over a list of hostnames.
- **Use SSH keys** (`key_filename="~/.ssh/id_ed25519"`) instead of passwords.
- **Schedule** it with cron, for example every 10 minutes: `*/10 * * * * python3 /opt/monitoring/memory_alert.py`.
- The same check-decide-ticket pattern works for **disk usage** (`df -h`), **CPU load** (`uptime`) and **failed services** (`systemctl is-failed nginx`).